<a href="https://colab.research.google.com/github/Muskan-Kandel/aes/blob/muskan%2Fdata-preprocessing/AES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install nltk transformers torch -q

# Install required libraries
import re
import json
import nltk
import torch
import pandas as pd
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from transformers import AutoTokenizer
from google.colab import drive
drive.mount('/content/drive')

#punkt and punkt_tab is used for tokenization
#(breaking paragraph into a list of words)
nltk.download('punkt')
nltk.download('punkt_tab')

#wordnet is a lexical database to group words into synonym sets
#and store relationships between words
nltk.download('wordnet')

#used for parts of speech tagging
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

Mounted at /content/drive


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [3]:
asap_raw = pd.read_csv('/content/drive/MyDrive/aes/training_set_rel3.csv', encoding='latin1')
asap_raw.shape

(12978, 28)

In [4]:
asap_raw.head()


,essay_id,essay_set,essay,rater1_domain1,rater2_domain1,rater3_domain1,domain1_score,rater1_domain2,rater2_domain2,domain2_score,...,rater2_trait3,rater2_trait4,rater2_trait5,rater2_trait6,rater3_trait1,rater3_trait2,rater3_trait3,rater3_trait4,rater3_trait5,rater3_trait6
0,1,1,"Dear local newspaper, I think effects computer...",4.0,4.0,NaN,8.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1,"Dear @CAPS1 @CAPS2, I believe that using compu...",5.0,4.0,NaN,9.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,1,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",4.0,3.0,NaN,7.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,1,"Dear Local Newspaper, @CAPS1 I have found that...",5.0,5.0,NaN,10.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,1,"Dear @LOCATION1, I know having computers has a...",4.0,4.0,NaN,8.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
prompt3 = pd.read_csv('/content/drive/MyDrive/aes/Prompt-3.csv', encoding='latin1')
prompt3.head()

,Essay ID,Content,Prompt Adherence,Language,Narrativity
0,5978,0,0,1,1
1,5979,3,3,2,2
2,5980,2,2,2,1
3,5981,0,1,0,0
4,5982,1,1,1,1


In [6]:
prompt4 = pd.read_csv('/content/drive/MyDrive/aes/Prompt-4.csv', encoding='latin1')
prompt4.head()

,Essay ID,Content,Prompt Adherence,Language,Narrativity
0,8863,0,0,0,0
1,8864,0,0,0,0
2,8865,3,3,3,3
3,8866,3,3,2,2
4,8867,2,2,2,2


In [7]:
prompt5 = pd.read_csv('/content/drive/MyDrive/aes/Prompt-5.csv', encoding='latin1')
prompt5.head()

,Essay ID,Content,Prompt Adherence,Language,Narrativity
0,11827,2,2,2,2
1,11828,3,3,3,3
2,11829,2,2,2,2
3,11830,1,0,1,1
4,11831,4,4,4,4


In [8]:
prompt6 = pd.read_csv('/content/drive/MyDrive/aes/Prompt-6.csv', encoding='latin1')
prompt6.head()

,Essay ID,Content,Prompt Adherence,Language,Narrativity
0,15491,4,3,2,2
1,15972,1,1,3,2
2,16109,4,4,3,3
3,16228,2,1,4,3
4,16352,3,3,4,4


In [9]:
filtered= asap_raw[asap_raw['essay_set'].isin([3,4,5,6])]
print("Total rows in prompt 3-6 is", len(filtered))
print("Columns : ", list(filtered.columns))
print(f"Shape: {filtered.shape}")
print("Per prompt : ", filtered['essay_set'].value_counts().sort_index().to_dict())
for i in [3,4,5,6]:
  p = pd.read_csv(f'/content/drive/MyDrive/aes/Prompt-{i}.csv')
  print(f'prompt-{i}.csv rows: {len(p)}, columns: {list(p.columns)}')

  #check for essay_id overlap in asap dataset and prompt 3-6 datasets

  overlap = filtered[filtered['essay_set'] == i]['essay_id'].isin(p['Essay ID']).sum()
  print(f'Matching essay_ids: {overlap}/{len(p)}')


Total rows in prompt 3-6 is 7103
Columns :  ['essay_id', 'essay_set', 'essay', 'rater1_domain1', 'rater2_domain1', 'rater3_domain1', 'domain1_score', 'rater1_domain2', 'rater2_domain2', 'domain2_score', 'rater1_trait1', 'rater1_trait2', 'rater1_trait3', 'rater1_trait4', 'rater1_trait5', 'rater1_trait6', 'rater2_trait1', 'rater2_trait2', 'rater2_trait3', 'rater2_trait4', 'rater2_trait5', 'rater2_trait6', 'rater3_trait1', 'rater3_trait2', 'rater3_trait3', 'rater3_trait4', 'rater3_trait5', 'rater3_trait6']
Shape: (7103, 28)
Per prompt :  {3: 1726, 4: 1772, 5: 1805, 6: 1800}
prompt-3.csv rows: 1726, columns: ['Essay ID', 'Content', 'Prompt Adherence', 'Language', 'Narrativity']
Matching essay_ids: 1726/1726
prompt-4.csv rows: 1772, columns: ['Essay ID', 'Content', 'Prompt Adherence', 'Language', 'Narrativity']
Matching essay_ids: 1772/1772
prompt-5.csv rows: 1805, columns: ['Essay ID', 'Content', 'Prompt Adherence', 'Language', 'Narrativity']
Matching essay_ids: 1805/1805
prompt-6.csv rows


This shows for every essay in filtered asap_raw data (Sets 3, 4, 5, and 6), there is a corresponding row in the Prompt-X.csv files.

In [10]:
filtered_clean = filtered[['essay_id', 'essay_set', 'essay', 'domain1_score']].copy()
trait_dfs = []
for i in [3, 4, 5, 6]:
    prompt_df = pd.read_csv(f'/content/drive/MyDrive/aes/Prompt-{i}.csv')

    # Standardize column names if necessary (e.g., 'Essay ID' to 'essay_id')
    prompt_df = prompt_df.rename(columns={'Essay ID': 'essay_id'})
    trait_dfs.append(prompt_df)

all_traits = pd.concat(trait_dfs, ignore_index = True)

#merge on essay_id
asap_plus_plus = filtered_clean.merge(all_traits, on='essay_id', how='inner')

#clean dataset
asap_plus_plus.dropna(subset=['essay', 'domain1_score'], inplace=True)
asap_plus_plus.drop_duplicates(subset=['essay'], inplace=True)
asap_plus_plus.reset_index(drop=True, inplace=True)

#check final dataset
print("Final columns:", list(asap_plus_plus.columns))
print("Final shape:", asap_plus_plus.shape)
print("Per prompt:\n", asap_plus_plus['essay_set'].value_counts().sort_index())
print("\nSample:")
asap_plus_plus.head(3)


Final columns: ['essay_id', 'essay_set', 'essay', 'domain1_score', 'Content', 'Prompt Adherence', 'Language', 'Narrativity']
Final shape: (7098, 8)
Per prompt:
 essay_set
3    1724
4    1769
5    1805
6    1800
Name: count, dtype: int64

Sample:


,essay_id,essay_set,essay,domain1_score,Content,Prompt Adherence,Language,Narrativity
0,5978,3,The features of the setting affect the cyclist...,1.0,0,0,1,1
1,5979,3,The features of the setting affected the cycli...,2.0,3,3,2,2
2,5980,3,Everyone travels to unfamiliar places. Sometim...,1.0,2,2,2,1


# **Score Normalization (Scale 0-10)**

Each prompt in ASAP++ has a different maximum score: Prompts 3 and 4 go up to **3** and Prompts 5 and 6 go up to **4**. This inconsistency can confuse the model, making it think a score of 3 means different things depending on the prompt.

To fix this, we scale all scores to a unified **0 to 10** range using:

$$Score_{norm} = \left( \frac{Score_{raw}}{S_{max}} \right) \times 10$$

| Prompt | Max Score | After Normalization |
| :--- | :--- | :--- |
| 3 | 3 | 10 |
| 4 | 3 | 10 |
| 5 | 4 | 10 |
| 6 | 4 | 10 |

This applies to all 5 score columns — `domain1_score`, `Content`, `Prompt Adherence`, `Language`, and `Narrativity`.

In [11]:
# Defining max scores per prompt
max_score_map = {3: 3, 4: 3, 5: 4, 6: 4}

# Trait columns to normalize
trait_cols = ['domain1_score', 'Content', 'Prompt Adherence', 'Language', 'Narrativity']

def normalize_scores(df):
    df = df.copy()
    for col in trait_cols:
        df[col] = df.apply(
            lambda row: round((row[col] / max_score_map[row['essay_set']]) * 10, 4),
            axis=1
        )
    return df

asap_plus_plus = normalize_scores(asap_plus_plus)

print("Scores normalized to 0-10 scale")
print("Sample scores after normalization:")
print(asap_plus_plus[['essay_set', 'domain1_score', 'Content', 'Prompt Adherence',
                        'Language', 'Narrativity']].head(5))


Scores normalized to 0-10 scale
Sample scores after normalization:
   essay_set  domain1_score  Content  Prompt Adherence  Language  Narrativity
0          3         3.3333   0.0000            0.0000    3.3333       3.3333
1          3         6.6667  10.0000           10.0000    6.6667       6.6667
2          3         3.3333   6.6667            6.6667    6.6667       3.3333
3          3         3.3333   0.0000            3.3333    0.0000       0.0000
4          3         6.6667   3.3333            3.3333    3.3333       3.3333


# **Text Normalization**

Raw essays from the ASAP dataset are messy. They contain encoding garbage like `ô` and `ö`, extra spaces, newlines and non-ASCII characters. We clean all of this so the model gets consistent, readable text.

**Steps:**
1. Fix encoding artifacts (`ô` -> `"`, `æ` ->`'`)
2. Remove HTML tags and URLs
3. Normalize whitespace and newlines
4. Lowercase everything
5. Remove non-ASCII characters
6. Fix punctuation spacing
7. Lemmatize : reduce words to base form (`running` -> `run`, `was` -> `be`)
8. Restore ASAP tokens (`@PERSON1`, `@NUM1`, `@CAPS1`) — these are preserved throughout and never altered

In [12]:
lemmatizer = WordNetLemmatizer()

# Pattern to detect ASAP anonymization tokens like @CAPS1, @NUM2, @PERSON
ASAP_TOKEN_PATTERN = re.compile(
    r'@(CAPS\d*|NUM\d*|PERSON\d*|ORGANIZATION\d*|LOCATION\d*|'
    r'DATE\d*|MONTH\d*|TIME\d*|MONEY\d*|PERCENT\d*|DR\d*|STATE\d*|CITY\d*)',
    re.IGNORECASE
)

def get_wordnet_pos(treebank_tag):
    """Map NLTK POS tag → WordNet POS tag so lemmatizer works correctly."""
    if treebank_tag.startswith('J'): return wordnet.ADJ
    if treebank_tag.startswith('V'): return wordnet.VERB
    if treebank_tag.startswith('N'): return wordnet.NOUN
    if treebank_tag.startswith('R'): return wordnet.ADV
    return wordnet.NOUN  # default

def normalize_essay(text):
    if not isinstance(text, str) or text.strip() == "":
        return ""

    asap_tokens = {}

    counter = [0]

    def replace_asap(match):
        key = f"ASAPTOKEN{counter[0]}"
        asap_tokens[key] = match.group(0).upper()
        counter[0] += 1
        return key
    text = ASAP_TOKEN_PATTERN.sub(replace_asap, text)

    # Fix encoding artifacts
    # ASAP dataset was saved in latin-1 which causes weird characters
    fixes = {
        'ô': '"', 'ö': '"', 'æ': "'", 'Æ': "'", 'ø': 'o',
        '\x93': '"', '\x94': '"', '\x91': "'", '\x92': "'",
        '\x85': '...', '\x96': '-', '\x97': '-',
        '\u2019': "'", '\u2018': "'", '\u201c': '"', '\u201d': '"',
    }
    for bad, good in fixes.items():
        text = text.replace(bad, good)

    # Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Normalize whitespace
    text = re.sub(r'[\t\n\r]+', ' ', text)

    # Lowercase
    text = text.lower()

    # Remove non-ASCII characters
    text = text.encode('ascii', 'ignore').decode('ascii')

    # Keep only useful characters
    text = re.sub(r"[^a-z0-9\s\.,!?'\"-]", '', text)

    # Fix punctuation spacing
    text = re.sub(r'\s([.,!?])', r'\1', text)
    text = re.sub(r'([.,!?])([^\s\d])', r'\1 \2', text)

    # Collapse spaces
    text = re.sub(r' +', ' ', text).strip()

    # Tokenize
    # We POS tag first so lemmatizer knows context
    # "running" as VERB → "run"  |  "running" as NOUN → "running"
    tokens = nltk.word_tokenize(text)
    pos_tags = nltk.pos_tag(tokens)
    lemmatized = [
        lemmatizer.lemmatize(word, get_wordnet_pos(pos))
        for word, pos in pos_tags
    ]
    text = ' '.join(lemmatized)

    # Restore ASAP tokens
    for key, original in asap_tokens.items():
        text = text.replace(key.lower(), original)

    return re.sub(r' +', ' ', text).strip()


print("Normalizing essay text")
asap_plus_plus['essay_normalized'] = asap_plus_plus['essay'].apply(normalize_essay)
print("Text normalizated")

Normalizing essay text
Text normalizated


In [13]:
# Quick check
print("\nBEFORE:", asap_plus_plus['essay'].iloc[1][:200])
print("\nAFTER: ", asap_plus_plus['essay_normalized'].iloc[1][:200])


BEFORE: The features of the setting affected the cyclist in a negative way. He was in the desert, which is dry and hot. ôI was traveling through the high deserts of California in June."(Kurmaskie @NUM1). Also

AFTER:  the feature of the setting affect the cyclist in a negative way . he be in the desert , which be dry and hot . `` i be travel through the high desert of california in june . `` kurmaskie @NUM1 . also 
